# Personal AI RAG

Conversational AI that knows everything about me, based on my knowledge base (career-history, blog).
Pipeline: load documents → chunk → vectorize in Chroma → chat via Gradio with OpenRouter.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr


In [ ]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 50
BASE_URL = "https://openrouter.ai/api/v1"

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_api_key:
    print(f"OpenAI API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openrouter = OpenAI(base_url=BASE_URL, api_key=openrouter_api_key)


In [ ]:
class Result(BaseModel):
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)

class Chunks(BaseModel):
    chunks: list[Chunk]

def fetch_documents():
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        if not folder.is_dir():
            continue
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents

def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the personal knowledge base of Hayatu Sanusi.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about Hayatu.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

def process_document(document):
    messages = [{"role": "user", "content": make_prompt(document)}]
    response = openrouter.beta.chat.completions.parse(model=MODEL, messages=messages, response_format=Chunks)
    doc_as_chunks = response.choices[0].message.parsed.chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

def create_chunks(documents):
    chunks = []
    for doc in documents:
        chunks.extend(process_document(doc))
    return chunks

documents = fetch_documents()
chunks = create_chunks(documents)
print(f"Created {len(chunks)} chunks")


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

lc_docs = [Document(page_content=r.page_content, metadata=r.metadata) for r in chunks]
vectorstore = Chroma.from_documents(documents=lc_docs, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=openrouter_api_key,
    model=MODEL,
    temperature=0,
)


In [ ]:
SYSTEM_PROMPT_TEMPLATE = """You are a friendly assistant that knows everything about Hayatu Sanusi (career, blog, experience).
Use the context below from Hayatu's knowledge base to answer questions. Be concise and accurate.
If the context doesn't contain the answer, say so.

Context:
{context}
"""

def format_chunks_for_display(docs):
    if not docs:
        return "*No chunks retrieved.*"
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "—")
        doc_type = doc.metadata.get("type", doc.metadata.get("doc_type", ""))
        header = f"Extract from {source}" + (f" ({doc_type})" if doc_type else "") + ":"
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


In [ ]:
chunk_display_handle = display(display_id=True)

def submit(message, history):
    if not message or not message.strip():
        return history, "", ""

    docs = retriever.invoke(message)
    chunks_display = format_chunks_for_display(docs)
    update_display(Markdown(chunks_display), display_id=chunk_display_handle.display_id)
    context = chunks_display
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    new_history = history + [{"role": "user", "content": message}]

    yield new_history + [{"role": "assistant", "content": ""}], "", chunks_display
    full = ""
    for chunk in llm.stream([SystemMessage(content=system_prompt), HumanMessage(content=message)]):
        if getattr(chunk, "content", None):
            full += chunk.content
            yield new_history + [{"role": "assistant", "content": full}], "", chunks_display

with gr.Blocks(title="Personal AI RAG") as demo:
    gr.Markdown("# Personal AI — Chat with your knowledge base")
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="Chat", type="messages", height=500)
            msg = gr.Textbox(
                placeholder="Ask anything about Hayatu...",
                show_label=False,
            )
        with gr.Column(scale=1):
            context_display = gr.Textbox(
                value="Retrieved chunks will appear here after you send a message.",
                label="Retrieved chunks (context used by the LLM)",
                interactive=False,
                lines=28,
            )
    msg.submit(submit, inputs=[msg, chatbot], outputs=[chatbot, msg, context_display])

demo.launch(inbrowser=True)
